In [1]:
# =============================================================================
# PATH SETUP - Add src folder to Python path
# =============================================================================

import sys
from pathlib import Path

# Add project root and src folder to Python path
project_root = Path.cwd().parent
src_path = project_root / 'src' / 'eeg_synthetic'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'✓ Added to Python path:')
print(f'  {src_path}')
print(f'  {project_root}')

✓ Added to Python path:
  /home/giorgio99/gan_bci/EEG-synthetic-main/EEG-synthetic-main/src/eeg_synthetic
  /home/giorgio99/gan_bci/EEG-synthetic-main/EEG-synthetic-main


## 1. Setup and Configuration

Import required libraries and configure paths.

In [3]:
# =============================================================================
# IMPORTS AND CONFIGURATION
# =============================================================================

# Standard libraries
import os
import numpy as np

# Custom modules for EEG data processing
from data_loader import BCIAUTLoader, plot_normalized_arrays
from complexity_metrics import calculate_complexity_metrics
from oversampling import apply_smote_3d

# Classification modules
from classifiers import (
    run_p300_classification,  # Main classification pipeline
    classify_eeg,              # Individual classifier wrapper
    EEGNetModel,               # Deep learning model
    TrainModel,                # Training utilities
    EvalModel                  # Evaluation utilities
)

# Scikit-learn classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# =============================================================================
# DATASET CONFIGURATION
# =============================================================================

# Get project root directory
project_root = os.getcwd()

# Build path to data folder
dataset_path = os.path.join(project_root, 'data')
print(f"Dataset path: {dataset_path}")

# Initialize the BCI-AUT data loader
loader = BCIAUTLoader(dataset_path)

# =============================================================================
# SCENARIO A: CROSS-SESSION (SAME SUBJECT)
# =============================================================================
# Train and test on different sessions from the same subject (Subject 3)

print("\n--- SCENARIO A: Cross-Session (Subject 3) ---")

# Load training data: Subject 3, Session 3, Training set
# Returns:
#   X_train_sess: EEG signals (trials, channels, timepoints)
#   y_train_sess: Labels (0=Nontarget, 1=P300)
X_train_sess, y_train_sess, subjects_train_ids, session_train_ids = loader.get_data(
    subjects='all', 
    sessions='all', 
    modes=['Train']
)

# Load test data: Subject 3, Session 3, Test set
X_test_sess, y_test_sess, subjects_test_ids, session_test_ids = loader.get_data(
    subjects='all', 
    sessions='all', 
    modes=['Test']
)

# =============================================================================
# SCENARIO B: CROSS-SUBJECT
# =============================================================================
# Note: This section is prepared but uses the same data as Scenario A
# To activate cross-subject validation, modify the subjects parameter above

print("\n--- SCENARIO B: Cross-Subject (Train: 1-14, Test: 15) ---")
print(f"\nDataset Cross-Subject ready:")
print(f"X_train shape: {X_train_sess.shape}")
print(f"X_test shape: {X_test_sess.shape}")


Dataset path: /home/giorgio99/gan_bci/EEG-synthetic-main/EEG-synthetic-main/notebooks/data

--- SCENARIO A: Cross-Session (Subject 3) ---
Loading - Subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], Sessions: [1, 2, 3, 4, 5, 6, 7], Modes: ['Train']


/home/giorgio99/gan_bci/EEG-synthetic-main/EEG-synthetic-main/src/eeg_synthetic/data_loader.py:179: RuntimeWarning: filter_length (8251) is longer than the signal (350), distortion is likely. Reduce filter length or filter a longer signal.
  epochs.filter(l_freq=0.1, h_freq=15.0, fir_design='firwin', verbose=False)
/home/giorgio99/gan_bci/EEG-synthetic-main/EEG-synthetic-main/src/eeg_synthetic/data_loader.py:179: RuntimeWarning: filter_length (8251) is longer than the signal (350), distortion is likely. Reduce filter length or filter a longer signal.
  epochs.filter(l_freq=0.1, h_freq=15.0, fir_design='firwin', verbose=False)
/home/giorgio99/gan_bci/EEG-synthetic-main/EEG-synthetic-main/src/eeg_synthetic/data_loader.py:179: RuntimeWarning: filter_length (8251) is longer than the signal (350), distortion is likely. Reduce filter length or filter a longer signal.
  epochs.filter(l_freq=0.1, h_freq=15.0, fir_design='firwin', verbose=False)
/home/giorgio99/gan_bci/EEG-synthetic-main/EEG-sy

KeyboardInterrupt: 

## 2. Data Loading and Exploration

Load EEG data and check class distribution.

In [3]:
if isinstance(y_train_sess, np.ndarray):
    classes, counts = np.unique(y_train_sess, return_counts=True)
else:
    classes, counts = torch.unique(y_train_sess, return_counts=True)
    classes, counts = classes.cpu().numpy(), counts.cpu().numpy()

print(f"X_train shape: {X_train_sess.shape}")
print("-" * 35)
for cls, count in zip(classes, counts):
    label = "Win (0)" if cls == 0 else "Loss (1)"
    print(f"Classe {int(cls)} [{label}]: {count} trial")

# Calcolo dello sbilanciamento in percentuale
total = counts.sum()
perc_0 = (counts[0] / total) * 100
perc_1 = (counts[1] / total) * 100
print("-" * 35)
print(f"Distribuzione: {perc_0:.2f}% (Win) vs {perc_1:.2f}% (Loss)")

X_train shape: (1600, 8, 128)
-----------------------------------
Classe 0 [Win (0)]: 1400 trial
Classe 1 [Loss (1)]: 200 trial
-----------------------------------
Distribuzione: 87.50% (Win) vs 12.50% (Loss)


In [ ]:
plot_normalized_arrays(X_train_sess, y_train_sess, X_test_sess, y_test_sess)

### 6.2 Load and save csv file

In [4]:
import pandas as pd
import numpy as np
import torch
import os

def save_to_eeg_csv(X, y, subject_ids, session_ids, filename):
    """
    Converte i tensori EEG in formato CSV compatibile con il tutorial.
    """
    n_trials, n_channels, n_times = X.shape
    
    # Appiattiamo i dati del segnale
    data_flat = X.reshape(-1, n_times)
    if isinstance(data_flat, torch.Tensor):
        data_flat = data_flat.numpy()

    # Preparazione colonne comuni
    condition_col = np.repeat(y if isinstance(y, np.ndarray) else y.numpy(), n_channels)
    trial_col = np.repeat(np.arange(1, n_trials + 1), n_channels)
    electrode_col = np.tile(np.arange(1, n_channels + 1), n_trials)
    
    # Metadati ID
    unique_subjs = np.unique(subject_ids)
    participant_col_raw = np.repeat(subject_ids, n_channels)
    session_col_raw = np.repeat(session_ids, n_channels)

    meta_data = {}
    # Logica ID: se c'è più di un soggetto usa ParticipantID, altrimenti SessionID
    if len(unique_subjs) > 1:
        meta_data['ParticipantID'] = participant_col_raw
    else:
        meta_data['ParticipantID'] = session_col_raw

    meta_data['Condition'] = condition_col
    meta_data['Trial'] = trial_col
    meta_data['Electrode'] = electrode_col

    # Assemblaggio
    df_meta = pd.DataFrame(meta_data)
    time_cols = [f'Time{i}' for i in range(1, n_times + 1)]
    df_data = pd.DataFrame(data_flat, columns=time_cols)
    df = pd.concat([df_meta, df_data], axis=1)

    # Salvataggio
    df.to_csv(filename, index=False)
    print(f"File salvato: {filename} | Shape: {df.shape}")

# --- ESECUZIONE ---

# Salva il Training Set
save_to_eeg_csv(X_train_sess, y_train_sess, subjects_train_ids, session_train_ids, 'dataset_eeg_train_3_subj_3_sess.csv')

# Salva il Test Set (Il file che ti serve per il codice del tutorial)
save_to_eeg_csv(X_test_sess, y_test_sess, subjects_test_ids, session_test_ids, 'dataset_eeg_test_3_subj_3_sess.csv')

File salvato: dataset_eeg_train_3_subj_3_sess.csv | Shape: (12800, 132)
File salvato: dataset_eeg_test_3_subj_3_sess.csv | Shape: (22400, 132)


In [5]:
import pandas as pd
import torch
import numpy as np
import os

def load_balanced_synthetic_limited(path_c0, path_c1, max_trials=1200):
    """
    Carica esplicitamente i primi 'max_trials' per classe 0 e classe 1.
    """
    # Calcoliamo quante righe leggere (8 righe per ogni trial)
    rows_to_read = max_trials * 8

    def get_data_from_csv(path):
        if not os.path.exists(path):
            print(f"ERRORE: File non trovato {path}")
            return None, None
        
        # Leggiamo solo le righe necessarie con nrows
        df = pd.read_csv(path, nrows=rows_to_read)
        
        # Prendiamo le colonne Time
        time_cols = [col for col in df.columns if col.startswith('Time')]
        X = df[time_cols].values
        y = df['Condition'].values
        return X, y

    # Carichiamo i due blocchi limitati
    print(f"Caricamento primi {max_trials} trial Classe 0...")
    X0_rows, y0_rows = get_data_from_csv(path_c0)
    
    print(f"Caricamento primi {max_trials} trial Classe 1...")
    X1_rows, y1_rows = get_data_from_csv(path_c1)

    # Conversione in Tensor e Raggruppamento
    # Se abbiamo letto 68800 righe, diventano 8600 trial da 8 elettrodi
    X0_trial = torch.tensor(X0_rows).view(-1, 8, 128)
    y0_trial = torch.tensor(y0_rows[::8]) 
    
    X1_trial = torch.tensor(X1_rows).view(-1, 8, 128)
    y1_trial = torch.tensor(y1_rows[::8])

    return X0_trial, y0_trial, X1_trial, y1_trial

# --- ESECUZIONE ---
file_c0 = 'generated_samples/aegan_data_c0_subj_3_sess_3.csv'
file_c1 = 'generated_samples/aegan_data_c1_subj_3_sess_3.csv'

# Carichiamo solo 8600 trials per classe
X0_syn, y0_syn, X1_syn, y1_syn = load_balanced_synthetic_limited(file_c0, file_c1, max_trials=8600)

print("-" * 30)
print(f"Trial caricate Classe 0: {X0_syn.shape[0]} (Labels: {y0_syn.shape[0]})")
print(f"Trial caricate Classe 1: {X1_syn.shape[0]} (Labels: {y1_syn.shape[0]})")

Caricamento primi 8600 trial Classe 0...
Caricamento primi 8600 trial Classe 1...
------------------------------
Trial caricate Classe 0: 1200 (Labels: 1200)
Trial caricate Classe 1: 1200 (Labels: 1200)
